In [1]:
import pandas as pd
import numpy as np

# 1. Veriyi Oku (ai4i2020.csv)
df = pd.read_csv('ai4i2020.csv')

# 2. Gereksiz kolonları at
df_clean = df.drop(['UDI', 'Product ID', 'Type'], axis=1)

# 3. Manuel Gruplandırma (Binning)
# KNIME'daki Numeric Binner mantığını kodla kuruyoruz
df_clean['Yuksek_Sicaklik'] = df_clean['Air temperature [K]'] > 301
df_clean['Yuksek_Tork'] = df_clean['Torque [Nm]'] > 50
df_clean['Asiri_Asinma'] = df_clean['Tool wear [min]'] > 180
df_clean['ARIZA'] = df_clean['Machine failure'] == 1

# 4. Kural Analizi (Basit ve Etkili Yöntem)
# Sadece ARIZA olan durumların diğer özelliklerle ilişkisine bakalım
ariza_durumlari = df_clean[df_clean['ARIZA'] == True]

# Toplam arıza sayısı
toplam_ariza = len(ariza_durumlari)
genel_ariza_orani = len(ariza_durumlari) / len(df_clean)

# Örnek bir kural analizi: Yüksek Tork varken Arıza olma olasılığı
y_tork_ariza = len(ariza_durumlari[ariza_durumlari['Yuksek_Tork'] == True])
y_tork_toplam = len(df_clean[df_clean['Yuksek_Tork'] == True])

confidence = y_tork_ariza / y_tork_toplam if y_tork_toplam > 0 else 0
lift = confidence / genel_ariza_orani if genel_ariza_orani > 0 else 0

print(f"--- Manuel Analiz Sonuçları ---")
print(f"Genel Arıza Oranı: %{genel_ariza_orani*100:.2f}")
print(f"Yüksek Tork Durumunda Arıza Olasılığı (Confidence): %{confidence*100:.2f}")
print(f"Yüksek Torkun Arıza Üzerindeki Etkisi (Lift): {lift:.2f} kat")

# Tüm kombinasyonları tablo olarak görelim
analiz_tablosu = df_clean.groupby(['Yuksek_Sicaklik', 'Yuksek_Tork', 'Asiri_Asinma'])['ARIZA'].mean().reset_index()
analiz_tablosu.columns = ['Y.Sıcaklık', 'Y.Tork', 'A.Aşınma', 'Arıza_Olasılığı']
print("\n--- Tüm Kombinasyonlar ve Arıza Riskleri ---")
print(analiz_tablosu.sort_values(by='Arıza_Olasılığı', ascending=False))

C:\Users\ibrah\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\ibrah\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


--- Manuel Analiz Sonuçları ---
Genel Arıza Oranı: %3.39
Yüksek Tork Durumunda Arıza Olasılığı (Confidence): %13.69
Yüksek Torkun Arıza Üzerindeki Etkisi (Lift): 4.04 kat

--- Tüm Kombinasyonlar ve Arıza Riskleri ---
   Y.Sıcaklık  Y.Tork  A.Aşınma  Arıza_Olasılığı
7        True    True      True         0.435897
3       False    True      True         0.350877
6        True    True     False         0.200000
5        True   False      True         0.050885
2       False    True     False         0.041480
1       False   False      True         0.037871
4        True   False     False         0.021455
0       False   False     False         0.004503
